# TechArena CTR: fixed numeric-choice and XGBoost ensemble
Attach both private prepared datasets. Reproduce the frozen XGBoost interaction expert and eight-epoch setwise numeric choice model; reuse the latter only after checkpoint and prediction replay.
Fixed80/10/10 rank blend, no blend search. Both dates must improve against V53 and each individual10% expert blend. Local reused development cohorts; no official-score forecast and no CSV export.


In [ ]:
from pathlib import Path
import hashlib, importlib.util
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
roots=list(Path("/kaggle/input").rglob("bundle_manifest.json"))
neural_roots=list(Path("/kaggle/input").rglob("neural_manifest.json"))
assert len(roots)==len(neural_roots)==1
DATA_ROOT=roots[0].parent
NEURAL_ROOT=neural_roots[0].parent
weighting_path=Path("/kaggle/working/tabular_choice_weighting.py")
weighting_path.write_text("\"\"\"Fixed-capacity CatBoost GPU expert on the existing causal feature bundle.\r\n\r\nNo evaluation labels enter fitting or early stopping. Main day6 is followed by\r\nunchanged day5 confirmation only if its conditional gain gate passes.\r\n\"\"\"\r\nimport gc\r\nimport importlib.util\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport catboost\r\nimport numpy as np\r\nfrom catboost import CatBoostClassifier, Pool\r\n\r\nPARAMS=dict(iterations=1600,depth=8,learning_rate=.04,l2_leaf_reg=10.,\r\n            loss_function=\"Logloss\",task_type=\"GPU\",devices=\"0\",border_count=63,\r\n            bootstrap_type=\"Bernoulli\",subsample=.8,random_seed=6801,\r\n            boosting_type=\"Plain\",thread_count=4,allow_writing_files=False,\r\n            verbose=100)\r\nBLEND=.25\r\n\r\n\r\ndef helpers(root):\r\n    ensemble_path=Path("/kaggle/working/choice_xgb_ensemble.py")
ensemble_path.write_text("\"\"\"Fixed 80/10/10 baseline/XGBoost/numeric-choice rank ensemble.\n\nReproduce the previously frozen XGBoost interaction recipe; validate every\nsaved neural and tree prediction before combining. No blend search.\n\"\"\"\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\nXGB_PARAMETERS = dict(max_depth=8, eta=.035, min_child_weight=100,\n                      reg_lambda=15., reg_alpha=.5)\nROUNDS = 1000\n\n\ndef module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    result = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(result)\n    return result\n\n\ndef combine(tree, neural, h):\n    for key in ['y', 'events', 'baseline', 'weights']:\n        assert np.array_equal(tree[key], neural[key]), 'Misaligned '+key\n    assert len(tree['y']) % 10 == 0\n    for raw in [tree['raw'], neural['raw']]:\n        assert raw.shape == tree['y'].shape\n        assert np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all()\n    p = h.rank(.8*h.rank(tree['baseline'])+.1*h.rank(tree['raw'])+.1*h.rank(neural['raw']))\n    return p\n\n\ndef run(root, neural_root, output, neural_source, weighting_source, neural_output):\n    root, output, neural_output = Path(root), Path(output), Path(neural_output)\n    assert not output.exists(), 'Choose a fresh experiment directory'\n    h = module('ensemble_tree', root/'runner.py')\n    neural = module('ensemble_numeric', neural_source)\n    assert neural.RECIPE['epochs'] == 8 and neural.RECIPE['blend_weight'] == .1\n    if not (neural_output/'summary.json').exists():\n        neural.run(root, Path(neural_root), neural_output, weighting_source)\n    neural_summary = json.loads((neural_output/'summary.json').read_text())\n    assert neural_summary['eligible_for_review']\n    h.PARAMS.update(XGB_PARAMETERS)\n    output.mkdir(parents=True)\n    recipe = dict(weights={'V53':.8,'xgboost':.1,'numeric_choice':.1},\n                  xgboost_params=h.PARAMS, rounds=ROUNDS, neural_recipe=neural.RECIPE,\n                  source_sha256=h.digest(Path(__file__)),\n                  neural_source_sha256=h.digest(Path(neural_source)),\n                  selection='Fixed before scoring, no blend search or validation-label fitting')\n    (output/'recipe.json').write_text(json.dumps(recipe, indent=2))\n    (output/'choice_xgb_ensemble.py').write_bytes(Path(__file__).read_bytes())\n    (output/'neural_source.py').write_bytes(Path(neural_source).read_bytes())\n    tree_output = output/'xgboost'\n    tree_output.mkdir()\n    tree_summary = h.run(root, tree_output, device='cuda', mode='main', rounds=ROUNDS, verify=True)\n    assert tree_summary['eligible_for_review'], 'Reproduced tree did not pass both date gates'\n    manifest = h.load_bundle(root, verify=False)\n    reports = []\n    for day in [6,5]:\n        directory = tree_output/f'day{day}'\n        with np.load(directory/'predictions.npz') as z:\n            tree = {k:z[k].copy() for k in z.files}\n        # Rerunning experiment must load the saved model and reproduce all rows.\n        h.experiment(root, manifest, directory, day=day, device='cuda', rounds=ROUNDS, mode='main')\n        with np.load(directory/'predictions.npz') as z:\n            assert np.array_equal(z['raw'], tree['raw'])\n            assert np.array_equal(z['pred'], tree['pred'])\n        ndir = neural_output/f'day{day}'\n        nr = json.loads((ndir/'report.json').read_text())\n        assert h.digest(ndir/'model.pt') == nr['model_sha256']\n        with np.load(ndir/'predictions.npz') as z:\n            nn = {k:z[k].copy() for k in z.files}\n        state = torch.load(ndir/'model.pt', map_location='cuda', weights_only=True)\n        model = neural.ChoiceNet(state['mean'], state['scale'], neural.RECIPE).to('cuda')\n        model.load_state_dict(state)\n        xv = h.FeatureBlocks(root, [manifest['validation'][str(day)]['features']])\n        replay = neural.predict(model, xv, 'cuda')\n        assert np.array_equal(replay, nn['raw'])\n        pred = combine(tree, nn, h)\n        result = h.metrics(tree['y'], pred, tree['baseline'], tree['weights'])\n        versus_tree = h.metrics(tree['y'], pred, tree['pred'], tree['weights'])\n        versus_neural = h.metrics(tree['y'], pred, nn['pred'], tree['weights'])\n        result.update(day=day, versus_tree=versus_tree, versus_neural=versus_neural,\n                      rows=len(pred), saved_checkpoint_replay='PASS, both experts exact',\n                      csv_ready=False, official_auc=None)\n        np.savez_compressed(output/f'day{day}.npz', pred=pred, tree_pred=tree['pred'],\n                            neural_pred=nn['pred'], neural_raw=nn['raw'],\n                            **{k:v for k,v in tree.items() if k != 'pred'})\n        reports.append(result)\n        print(json.dumps(result, indent=2), flush=True)\n    eligible = all(r['positive_gate'] and r['versus_tree']['positive_gate']\n                   and r['versus_neural']['positive_gate'] for r in reports)\n    summary = dict(reports=reports, eligible_for_review=eligible, target=.83,\n                   target_achieved=bool(eligible and reports[0]['history_adjusted_auc'] >= .83),\n                   csv_ready=False, official_auc=None,\n                   note='Reused development dates; intervals omit fitting and repeated selection uncertainty.')\n    (output/'summary.json').write_text(json.dumps(summary, indent=2))\n    return summary\n",encoding="utf-8")
assert hashlib.sha256(ensemble_path.read_bytes()).hexdigest()=="23e932da49f5e1b9f03efef3c3418632e25b0778fe86198d36b266e595fa1e9a"
spec=importlib.util.spec_from_file_location("ctr_ensemble",ensemble_path)
ensemble=importlib.util.module_from_spec(spec)
spec.loader.exec_module(ensemble)
RUN_NAME="ctr-choice-xgb-ensemble-001"
output=Path("/kaggle/working")/RUN_NAME
print("Frozen blend weights: V53 .8, XGBoost .1, eight-epoch setwise neural .1")
summary=ensemble.run(DATA_ROOT,NEURAL_ROOT,output,source_path,weighting_path,Path("/kaggle/working/ctr-tabular-choice-003-setwise"))
summary

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json, zipfile
print("Experiment:", RUN_NAME)
report = json.loads((output / "summary.json").read_text())
for result in report["reports"]:
    print("Day", result["day"], "Adjusted AUC:", round(result["history_adjusted_auc"], 9), "Ordinary AUC:", round(result["ordinary_auc"], 9), "Gate:", result["positive_gate"])
print("Target 0.83 reached:", report["target_achieved"])
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
